### COPY INTO and MERGE Commands




#### COPY INTO Command

- Incrementally loads data into Delta lake tables using Cloud Storage 
- Supports Schema Evolution
- Support Wide range of file Formats (CSV, JSON, Parquet, Delta)
- Alternative to Auto Loader for batch ingestion

##### Create a table to copy the data into

In [0]:
%sql
create table if not exists gtesting.delta_lake.raw_stock_prices;

##### Incrementally Load new files into the table

DOCUMENTATION - [COPY_INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-copy-into)

In [0]:
%sql
-- DELETE FROM gtesting.delta_lake.raw_stock_prices;

COPY INTO gtesting.delta_lake.raw_stock_prices
FROM 'abfss://gizmobox@dbstorageacc123.dfs.core.windows.net/landing/stock_prices'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema'='true')
COPY_OPTIONS ('mergeSchema'='true');

In [0]:
%sql
select * from gtesting.delta_lake.raw_stock_prices;

### MERGE STATEMENT

- Used for upserts(Insert/update/Delete Operations in a single Statement)
- Allow merging new Data into target table based on matching condition


Documentation - [MERGE_INTO](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/delta-merge-into)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gtesting.delta_lake.stock_prices
(
    stock_id STRING,
    price DOUBLE,
    trading_date STRING
);


In [0]:
%sql
MERGE INTO gtesting.delta_lake.stock_prices as target
USING gtesting.delta_lake.raw_stock_prices as source
ON target.stock_id = source.stock_id
WHEN MATCHED AND source.status = "ACTIVE" THEN
      UPDATE SET target.price = source.price,target.trading_date = source.trading_date
WHEN  MATCHED AND source.status = "DELISTED" THEN
      DELETE
WHEN NOT MATCHED AND source.status = "ACTIVE" THEN
    INSERT (stock_id,price,trading_date) VALUES (source.stock_id,source.price,source.trading_date);



In [0]:
%sql
select * from gtesting.delta_lake.stock_prices;